# Task 03 — Promptable Segmentation (SAM / SAM2 / SAM2.1)

**Dataset:** COCO val2017 400 (GT-box prompts) — smoke/eval (30 instances)

**Protocol:** smoke_eval

**Models:** SAM, SAM2, SAM2.1, MobileSAM, MedSAM

In [ ]:
# v2.49.0: Foundation/promptable segmentation contract-passed models.
# These models load, run, and produce valid mask output (contract_passed).
# Full benchmark requires promptable segmentation dataset with GT masks.
PROMPTABLE_SEG_CONTRACT_MODELS = [
    # SAM family — contract_passed, output schema validated
    "sam-vit-base", "sam-vit-large", "sam-vit-huge",
    "sam2-hiera-base-plus", "sam2-hiera-large", "sam2-hiera-small",
    "sam2-hiera-tiny",
    "sam2.1-hiera-base-plus", "sam2.1-hiera-large", "sam2.1-hiera-small",
    "sam2.1-hiera-tiny",
    "edgesam",  # v2.57: EdgeSAM benchmarked (mask_mAP50:95=0.149)
    # Distilled SAM family
    "efficientsam", "hq-sam", "mobilesam", "medsam",
]
print(f"Promptable segmentation contract models: {len(PROMPTABLE_SEG_CONTRACT_MODELS)}")
print("dataset_prepare_command: visionservex dataset prepare-promptable-subset --source coco-instance:/home/arash/datasets/coco_val2017_400_vsx/annotations.json --out /home/arash/datasets/promptable_segmentation_vsx")


In [ ]:
import json, sys, os
from pathlib import Path

# Add shared dir to path
NB_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/home/arash/PycharmProjects/VisionServeX/notebook")
sys.path.insert(0, str(NB_ROOT / "shared"))
os.chdir(str(NB_ROOT.parent))

from paths import COCO_400_ANN, COCO_400_IMAGES, SMOKE_IMG, SMOKE_ANN, NB_ROOT, REPO_ROOT
from display import clean, scan_text
from commands import run
from notebook_utils import write_status

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import visionservex
print(f"VisionServeX {visionservex.__version__}")

TASK = NB_ROOT / '03_promptable_segmentation'


In [ ]:
if COCO_400_ANN.exists():
    result = run(["benchmark-promptable-segmentation",
                  "--dataset", str(COCO_400_ANN),
                  "--images-dir", str(COCO_400_IMAGES),
                  "--models", "sam2-hiera-tiny,sam2.1-hiera-tiny",
                  "--prompt-source", "gt-box",
                  "--max-instances", "30",
                  "--device", "cuda", "--format", "json",
                  "--out", str(TASK / "reports/promptable_eval.json")])
else:
    result = {"status":"expected_blocker","code":"COCO_INSTANCE_DATASET_REQUIRED"}

print(f"status : {result.get('status')}")
print(f"code   : {result.get('code','-')}")
for row in result.get("rows",[]):
    print(f"  {row.get('model_id')}: {row.get('status')}")


In [ ]:
write_status(TASK, task='03_promptable_segmentation', dataset='COCO val2017 400 (GT-box prompts) — smoke/eval (30 instances)', protocol='smoke_eval')
print('Status written.')